# Verify / fix Kilroy protocol consistency

A Kilroy config has three sections: **valve commands** (`<valve_cmd>`), **pump
commands** (`<pump_cmd>`), and **protocols** (`<protocol>`). Each protocol step is a
`<valve>` or `<pump>` element whose *text* names one of those commands. If a step
names a command that isn't defined — because of a typo or an inconsistent name —
Kilroy errors at load.

This notebook processes **every Kilroy config it finds** (loop over all configs):
1. reads the config,
2. **verifies** that every protocol step references a defined valve/pump command,
3. proposes fuzzy-matched corrections for any mismatch, and
4. after you confirm, **rewrites** the config in place (backing up to `*.bak`),
   preserving its exact line endings and ISO-8859-1 encoding.

The verify/fix logic lives in `MERci.acquisition.kilroy`
(`check_kilroy_consistency`, `format_consistency_report`, `fix_kilroy_consistency`)
so it can also be called from other code.

In [ ]:
import os
import sys
from pathlib import Path

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.kilroy import (
    load_kilroy_commands,
    iter_protocol_references,
    check_kilroy_consistency,
    format_consistency_report,
    fix_kilroy_consistency,
)

print(f"MERCI_DIR  : {MERCI_DIR}")
print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## Find the Kilroy configs

Collect every `kilroy-config-*.xml` to check. By default we look in the
experiment's `settings/` folder (where `prepare_imaging/03` copies the chosen
config) and in the repo's `data/configs/kilroy/` templates. Edit `KILROY_CONFIGS`
to restrict the list (e.g. to a single file) if you don't want to process all of
them.

In [ ]:
# Candidate locations, in priority order. De-duplicate while keeping order.
search_dirs = [SAMPLE_DIR / "settings", MERCI_DIR / "data" / "configs" / "kilroy"]

KILROY_CONFIGS = []
seen = set()
for d in search_dirs:
    for p in sorted(d.glob("kilroy-config-*.xml")):
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            KILROY_CONFIGS.append(p)

print(f"Found {len(KILROY_CONFIGS)} Kilroy config(s) to verify:")
for p in KILROY_CONFIGS:
    print(f"  {p}")

assert KILROY_CONFIGS, "No Kilroy configs found - set KILROY_CONFIGS manually."

# To verify only some, overwrite the list, e.g.:
#   KILROY_CONFIGS = [MERCI_DIR / "data/configs/kilroy/kilroy-config-mf3-hamilton-adaptors-and-direct-260619.xml"]

## Verify consistency (all configs)

Loop over every config, load its commands, and check every protocol step. Results
are stored in `issues_by_config` keyed by config path for the review/apply steps
below.

In [ ]:
issues_by_config = {}

for cfg in KILROY_CONFIGS:
    commands   = load_kilroy_commands(cfg)
    references = iter_protocol_references(cfg)
    issues     = check_kilroy_consistency(cfg)
    issues_by_config[cfg] = issues

    print("=" * 78)
    print(cfg.name)
    print(f"  valve commands={len(commands['valve'])}  "
          f"pump commands={len(commands['pump'])}  protocol steps={len(references)}")
    print(format_consistency_report(issues))
    print()

n_bad = sum(1 for v in issues_by_config.values() if v)
print("=" * 78)
print(f"Summary: {n_bad} of {len(KILROY_CONFIGS)} config(s) have inconsistencies.")

## Review and choose fixes (per config)

Each mismatch comes with the closest-matching defined command as a *suggestion*:

- **`[normalized]`** — the reference differs from the suggestion only by letter
  case and/or surrounding whitespace. These are almost always safe to apply.
- **others** — the nearest string match by similarity score. Treat these as
  hints, not answers: a low score, or an issue like *`Set Hyb 23` → `Set Hyb 2`*,
  usually means the command is genuinely **missing** from the config (you must add
  the command definition), not that the protocol has a typo. Verify before applying.

The cell below builds `fixes_by_config[cfg]` for each config, starting with **only
the safe, normalized matches**. Add reviewed fuzzy matches by hand, or remove any
you don't want, before running the apply cell.

In [ ]:
fixes_by_config = {}

for cfg, issues in issues_by_config.items():
    print("=" * 78)
    print(cfg.name)
    if not issues:
        print("  (consistent - nothing to fix)")
        fixes_by_config[cfg] = {}
        continue

    for i in issues:
        if i.suggestion is None:
            note = "no commands of this kind defined"
        elif i.normalized_match:
            note = "SAFE - case/whitespace only"
        else:
            note = f"REVIEW - nearest match, similarity {i.score:.2f}"
        print(f"  [{i.kind}] {i.referenced!r} -> {i.suggestion!r:30}  {note}")

    # Default: only the safe, normalized matches for this config.
    fixes_by_config[cfg] = {
        (i.kind, i.referenced): i.suggestion
        for i in issues
        if i.normalized_match
    }

# ── Manual overrides ───────────────────────────────────────────────────────
# To apply a reviewed fuzzy match, add it under the right config, e.g.:
#   cfg = KILROY_CONFIGS[0]
#   fixes_by_config[cfg][("valve", "SetReadouts")] = "Set Readouts"
# To drop a proposed fix:
#   del fixes_by_config[cfg][("valve", "Readouts")]

print("\n" + "=" * 78)
print("Fixes queued per config:")
for cfg, fixes in fixes_by_config.items():
    print(f"  {cfg.name}: {len(fixes)} fix(es)")
    for (kind, wrong), right in fixes.items():
        print(f"      [{kind}] {wrong!r} -> {right!r}")

## Apply fixes (all configs)

Loops over every config and rewrites its queued steps. Each config is backed up to
`<config>.bak` before its first change. Any queued fix that matches nothing is
flagged (usually a wrong `kind` or a stale name). A final re-check confirms each
result is clean.

In [ ]:
any_applied = False

for cfg, fixes in fixes_by_config.items():
    if not fixes:
        continue
    any_applied = True
    print("=" * 78)
    print(cfg.name)
    total, applied = fix_kilroy_consistency(cfg, fixes, backup=True)
    print(f"  Applied {total} replacement(s).")
    if total:
        print(f"  Backup written: {cfg.name}.bak")
    for kind, wrong, right, n in applied:
        flag = "" if n else "   <-- MATCHED NOTHING (check the name/kind)"
        print(f"    [{kind}] {wrong!r} -> {right!r}: {n} replaced{flag}")
    print("  Re-check:")
    print("   ", format_consistency_report(check_kilroy_consistency(cfg)).replace("\n", "\n    "))
    print()

if not any_applied:
    print("No fixes queued for any config - nothing to apply.")